# Cycle 1 — Tuning (Chronological Split)

Same `RandomizedSearchCV` configuration as `notebooks/cycle1_tuning.ipynb`. Only the train/test split is changed to chronological. Cross-validation is still `StratifiedKFold(n_splits=5)` over the **training** portion (which is itself contiguous in time) so the inner CV remains a defensible model-selection procedure on past data.

In [3]:
import sys, os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier


# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing

## Prepare data

In [4]:
#  chronological by Season
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED)).sort_values('Season').reset_index(drop=True)
split_idx1 = int(len(df1) * 0.8)

# Drop FTR (target) and Season (metadata used for splitting, not a feature)
X1_train = df1.iloc[:split_idx1].drop(columns=['FTR', 'Season'])
y1_train = df1.iloc[:split_idx1]['FTR']
X1_test  = df1.iloc[split_idx1:].drop(columns=['FTR', 'Season'])
y1_test  = df1.iloc[split_idx1:]['FTR']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {len(X1_train)} matches | Test: {len(X1_test)} matches')
print(f'Features ({X1_train.shape[1]}): {list(X1_train.columns)}')


Train: 5472 matches | Test: 1368 matches
Features (33): ['HomeTeam', 'AwayTeam', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']


## XGBoost search

In [7]:
xgb_param_grid = {
    'n_estimators':     [100,200,300,500],
    'max_depth':        [3,4,5,6],
    'learning_rate':    [0.01,0.05,0.1,0.2],
    'subsample':        [0.7,0.8,1.0],
    'colsample_bytree': [0.7,0.8,1.0],
    'min_child_weight': [1,3,5],
    'gamma':            [0,0.1,0.2],
}

xgb1 = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)
search_d1 = RandomizedSearchCV(xgb1, xgb_param_grid, n_iter=50, cv=cv,
                                scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_d1.fit(X1_train, y1_train)

print('Best params:', search_d1.best_params_)
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')
y_pred_xgb_d1 = search_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_xgb_d1, target_names=['Away Win','Draw','Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best CV accuracy: 53.02%
Test accuracy:    52.85%

              precision    recall  f1-score   support

    Away Win       0.54      0.44      0.49       406
        Draw       0.28      0.04      0.07       348
    Home Win       0.54      0.86      0.66       614

    accuracy                           0.53      1368
   macro avg       0.45      0.45      0.41      1368
weighted avg       0.47      0.53      0.46      1368



## Random Forest search

In [5]:
rf_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [10, 15, 20, 25],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
    'bootstrap':         [True, False],
    'class_weight':      ['balanced', 'balanced_subsample']
}

rf1 = RandomForestClassifier(random_state=42)
search_rf_d1 = RandomizedSearchCV(rf1, rf_param_grid, n_iter=50, cv=cv,
                                   scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_rf_d1.fit(X1_train, y1_train)

print('Best params (Random Forest):', search_rf_d1.best_params_)
print(f'Best CV accuracy: {search_rf_d1.best_score_*100:.2f}%')
y_pred_rf_d1 = search_rf_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_rf_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_rf_d1, target_names=['Away Win','Draw','Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params (Random Forest): {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20, 'class_weight': 'balanced_subsample', 'bootstrap': True}
Best CV accuracy: 51.26%
Test accuracy:    51.02%

              precision    recall  f1-score   support

    Away Win       0.48      0.46      0.47       406
        Draw       0.28      0.12      0.17       348
    Home Win       0.57      0.76      0.65       614

    accuracy                           0.51      1368
   macro avg       0.44      0.45      0.43      1368
weighted avg       0.47      0.51      0.48      1368



## LightGBM search

In [6]:
lgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'learning_rate':    [0.01, 0.05, 0.1, 0.15],
    'max_depth':        [3, 4, 5, 6, 7],
    'num_leaves':       [20, 31, 50, 100],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples':[10, 20, 30],
    'reg_lambda':       [0, 0.1, 1.0]
}

lgb1 = LGBMClassifier(random_state=42, verbose=-1)
search_lgb_d1 = RandomizedSearchCV(lgb1, lgb_param_grid, n_iter=50, cv=cv,
                                    scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_lgb_d1.fit(X1_train, y1_train)

print('Best params (LightGBM):', search_lgb_d1.best_params_)
print(f'Best CV accuracy: {search_lgb_d1.best_score_*100:.2f}%')
y_pred_lgb_d1 = search_lgb_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_lgb_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_lgb_d1, target_names=['Away Win','Draw','Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params (LightGBM): {'subsample': 0.8, 'reg_lambda': 1.0, 'num_leaves': 100, 'n_estimators': 500, 'min_child_samples': 10, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best CV accuracy: 53.25%
Test accuracy:    52.05%

              precision    recall  f1-score   support

    Away Win       0.53      0.43      0.48       406
        Draw       0.26      0.05      0.08       348
    Home Win       0.53      0.85      0.65       614

    accuracy                           0.52      1368
   macro avg       0.44      0.44      0.40      1368
weighted avg       0.46      0.52      0.46      1368



## Side-by-side comparison: Chronological vs Random tuning

- On random split we've got LightGBM as the best model with $54.61 \%$ accuracy, While here using chronological split the best one is XGBoost tunned with $52.85 \%$.

- Eventhough chronological split show accuracy decreasing with roughly $2 \%$ it still the best choice to use since we are dealing with time-series data. 

- Also we noticed that XGboost do better whith chronological splitting than random split.


In [8]:
from sklearn.metrics import accuracy_score

chrono_acc = accuracy_score(y1_test, y_pred_xgb_d1) * 100
comp = pd.DataFrame([
    {'Split': 'Random 80/20',  'XGBoost Tuned accuracy %': 52.78},
    {'Split': 'Chronological', 'XGBoost Tuned accuracy %': round(chrono_acc, 2)},
])
print(comp.to_string(index=False))

        Split  XGBoost Tuned (D1) accuracy %
 Random 80/20                          52.78
Chronological                          52.85


## Save the deployed Cycle 1 model

This notebook produces the **deployed** Cycle 1 model — chronological tuning is the honest evaluation, so its winner is what the API serves.


In [17]:
import joblib

best_xgb = search_d1.best_estimator_

joblib.dump(best_xgb,                str(Paths.C1_MODEL))
joblib.dump(list(X1_train.columns),  str(Paths.C1_FEATURES))

print(f'Model saved    -> {Paths.C1_MODEL}')
print(f'Features saved -> {Paths.C1_FEATURES}')
print(f'\nDeployed feature columns ({len(X1_train.columns)}): {list(X1_train.columns)}')

Model saved    -> /Users/mac/Desktop/diaa/freelance/football project/FootballPredictor/models/cycle1/cycle1_xgb_best.pkl
Features saved -> /Users/mac/Desktop/diaa/freelance/football project/FootballPredictor/models/cycle1/cycle1_feature_cols.pkl

Deployed feature columns (33): ['HomeTeam', 'AwayTeam', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']
